In [ ]:
!pip install -q torch-geometric

import time
import psutil
import torch
from torch_geometric.datasets import MD17
from torch_geometric.loader import DataLoader

print(f"System RAM Initial: {psutil.virtual_memory().percent}%")

# 1. Hardware Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Target Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 2. Fetch the Standard Benchmark Dataset
print("\nFetching MD17 Dataset (Aspirin trajectory)...")
dataset = MD17(root='./data', name='aspirin')
print(f"Total Graphs (Molecules) Loaded: {len(dataset)}")

# 3. The Naive DataLoader (The bottleneck we are targeting)
# Setting num_workers=4 triggers Python's copy-on-read RAM spike
batch_size = 64
num_workers = 4 
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

print(f"\nStarting Profiling Run (Batch Size: {batch_size}, Workers: {num_workers})...")

num_batches_to_profile = 500
cpu_load_times = []
gpu_compute_times = []

# Warm-up CUDA
if torch.cuda.is_available():
    torch.cuda.synchronize()
    
global_start = time.time()
epoch_start = time.time()

for idx, batch in enumerate(loader):
    if idx >= num_batches_to_profile:
        break
        
    # Time the CPU Data Loading
    load_end = time.time()
    cpu_load_times.append(load_end - epoch_start)
    
    # Move ragged graph to GPU
    batch = batch.to(device)
    
    # Dummy GPU workload (Matrix math simulating an AI neural network pass)
    if torch.cuda.is_available():
        dummy_matrix = torch.randn((batch_size, 512, 512), device=device)
        _ = torch.bmm(dummy_matrix, dummy_matrix)
        torch.cuda.synchronize()
    
    # Time the GPU compute
    compute_end = time.time()
    gpu_compute_times.append(compute_end - load_end)
    
    # Reset clock for next CPU load
    epoch_start = time.time()

global_end = time.time()

# 4. Results & Metrics extraction
total_cpu_time = sum(cpu_load_times)
total_gpu_time = sum(gpu_compute_times)
total_runtime = total_cpu_time + total_gpu_time

print("\n" + "="*50)
print("BASELINE METRICS (NAIVE PIPELINE)")
print("="*50)
print(f"Total Batches Processed : {num_batches_to_profile}")
print(f"Total Profiling Time    : {total_runtime:.2f} seconds")
print("-" * 50)
print(f"CPU Load Time (Idle GPU): {total_cpu_time:.2f} sec ({(total_cpu_time/total_runtime)*100:.1f}%)")
print(f"GPU Active Compute Time : {total_gpu_time:.2f} sec ({(total_gpu_time/total_runtime)*100:.1f}%)")
print("-" * 50)
print(f"System RAM Peak Used    : {psutil.virtual_memory().percent}%")
if torch.cuda.is_available():
    print(f"GPU VRAM Peak Used      : {torch.cuda.max_memory_allocated() / (1024 ** 2):.1f} MB")
print("="*50)
